In [6]:
# Load env variables and create client
from dotenv import load_dotenv
from rich.console import Console  # only for fancy text formatting
from anthropic import Anthropic

load_dotenv(override=True)
console = Console(force_jupyter=False)

client = Anthropic()
MODEL = "claude-haiku-4-5"
MAX_TOKENS = 1024

In [7]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": MODEL,
        "max_tokens": MAX_TOKENS,
        "messages": messages,
        "stop_sequences": stop_sequences,
        # this will work with older (<1.1.0) SDK
        # "temperature": temperature,
        # ------------------------------
        # for 1.1.0+ SDK use the following
        "extra_body": {"temperature": temperature},
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [8]:
import json


def generate_dataset():
    prompt = """
        Generate a evaluation dataset for a prompt evaluation. The dataset will be used
        to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related 
        tasks. Generate an array of JSON objects, each representing task that requires Python, 
        JSON, or a Regex to complete. 

        Example output:
        ```json
        [
            {
                "task": "Description of task",
            },
            ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function, a single 
          JSON object, or a regular expression.
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    response = chat(messages, stop_sequences=["```"])
    return json.loads(response)

In [9]:
# let's test the function defined above

dataset = generate_dataset()
console.print(dataset)

# write the dataset to JSON file
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

[
    {
        'task': "Write a Python function that extracts the AWS region from an 
S3 bucket ARN (e.g., 'arn:aws:s3:::my-bucket'). The function should return the 
region if present, or 'us-east-1' as default if the ARN doesn't contain region 
information."
    },
    {
        'task': "Create a JSON object representing an AWS IAM policy that 
allows read-only access to all objects in an S3 bucket named 'my-data-bucket'. 
The policy should include the appropriate actions (s3:GetObject, s3:ListBucket)
and resources."
    },
    {
        'task': "Write a regular expression that matches valid AWS EC2 security
group IDs. Valid IDs follow the format 'sg-' followed by exactly 8 or 17 
hexadecimal characters (e.g., 'sg-0a1b2c3d' or 'sg-0a1b2c3d4e5f6a7b8')."
    }
]


### Running the Evals

The `run_prompt` function below is not defining any formatting instructions, so expect a lot of text to be returned from Claude.

In [10]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the
    result"""

    # NOTE: test_case is one of JSON element from the sample
    # JSON above
    prompt = f"""
        Please solve the following task:

        {test_case["task"]}
    """

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [11]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
    }

In [12]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [15]:
# open the test-cases JSON and run the evaluations
import json

with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
console.print(results)

[
    {
        'output': '# AWS S3 Bucket ARN Region Extractor\n\nHere\'s a 
comprehensive solution with detailed explanation and 
tests:\n\n```python\nimport re\nfrom typing import Optional\n\ndef 
extract_region_from_s3_arn(arn: str) -> str:\n    """\n    Extracts the AWS 
region from an S3 bucket ARN.\n    \n    S3 ARN format: 
arn:aws:s3:::[region:]bucket-name\n    Note: S3 bucket ARNs typically don\'t 
include region, but this function\n    handles various formats including 
regional bucket references.\n    \n    Args:\n        arn: The S3 bucket ARN 
string\n        \n    Returns:\n        The AWS region if present, otherwise 
\'us-east-1\' as default\n        \n    Raises:\n        ValueError: If the ARN
is not a valid S3 ARN\n    """\n    \n    # Validate basic ARN format\n    if 
not isinstance(arn, str) or not arn.startswith(\'arn:aws:s3:::\'):\n        
raise ValueError(f"Invalid S3 ARN format: {arn}")\n    \n    # Extract the part
after \'arn:aws:s3:::\'\n    bucket_part =